# Índice Oceánico de El Niño (ONI) – Serie mensual con rezagos

Convierte el archivo `raw_data/oni.ascii.txt` (NOAA CPC) en una serie mensual lista para cruzar con los precios SIPSA:

`processed_data/oni_mensual.csv`

**Formato original:** el ONI es la anomalía de la temperatura superficial del mar en la región Niño 3.4,
promediada en **trimestres móviles** (`DJF` = diciembre-enero-febrero, `JFM` = enero-febrero-marzo, …).
La columna `YR` corresponde al año del **mes central** del trimestre (por eso `NDJ 1950` = nov-1950, dic-1950, ene-1951).

**Asignación al último mes (evita filtración de información futura):**
si el trimestre se asignara a su mes central, el valor de enero (DJF) incluiría datos de febrero, que en enero todavía
no se conocen (*data leakage*), y los modelos de pronóstico parecerían mejores de lo que realmente son.
Por eso cada trimestre se asigna a su **último mes**, cuando el dato ya está completo: `DJF` → febrero, `JFM` → marzo, …, `NDJ` → enero del año siguiente.

**Rezagos:** se agregan `ONI_lag1`, `ONI_lag3`, `ONI_lag6` y `ONI_lag12` (ONI de t−1, t−3, t−6 y t−12) para pronosticar el precio en t.
Además de evitar la filtración, tienen sentido físico: el efecto de El Niño / La Niña sobre cosechas y pastos
tarda varios meses en llegar a los precios (corto plazo para el arroz, largo plazo para la ganadería).
La elección de estos rezagos se respalda con el análisis de correlación cruzada de la sección 6.

> **Recomendación para el modelo:** usar al menos `ONI_lag1`. El CPC publica el valor de un trimestre a inicios del mes
> siguiente (p. ej. `JAS` se publica en octubre), así que en el mes t aún no se conoce el `ONI` asignado a t.
> La columna `ONI` (sin rezago) sirve para análisis descriptivo, no como predictor.

## 1. Preparar el entorno

In [ ]:
import os
import sys

REPO_URL = "https://github.com/yevasa93/AnomaliasClimaticasArrozCarne.git"
REPO_DIR = "AnomaliasClimaticasArrozCarne"

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    # En Colab se clona el repositorio público (o se actualiza si ya existe)
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    else:
        !git -C {REPO_DIR} pull
    os.chdir(REPO_DIR)

print("Directorio de trabajo:", os.getcwd())

In [ ]:
import pandas as pd

RAW_FILE = os.path.join("raw_data", "oni.ascii.txt")
OUT_DIR = "processed_data"
OUT_FILE = os.path.join(OUT_DIR, "oni_mensual.csv")

os.makedirs(OUT_DIR, exist_ok=True)

## 2. Lectura del archivo original

El archivo separa las columnas con espacios, por eso se lee con `sep=r"\s+"` (no hace falta convertirlo a CSV antes).

In [ ]:
oni_raw = pd.read_csv(RAW_FILE, sep=r"\s+")
print(oni_raw.shape)
oni_raw.head(13)

## 3. Asignación de cada trimestre a su último mes

| Trimestre | Último mes | Año |
|---|---|---|
| DJF | 2 (febrero) | YR |
| JFM | 3 (marzo) | YR |
| … | … | … |
| OND | 12 (diciembre) | YR |
| NDJ | 1 (enero) | **YR + 1** |

In [ ]:
ULTIMO_MES = {
    "DJF": 2, "JFM": 3, "FMA": 4, "MAM": 5, "AMJ": 6, "MJJ": 7,
    "JJA": 8, "JAS": 9, "ASO": 10, "SON": 11, "OND": 12, "NDJ": 1,
}

oni = oni_raw.rename(columns={"SEAS": "Trimestre", "TOTAL": "SST_Nino34", "ANOM": "ONI"})

desconocidos = set(oni["Trimestre"]) - set(ULTIMO_MES)
assert not desconocidos, f"Trimestres no reconocidos: {desconocidos}"

oni["Mes"] = oni["Trimestre"].map(ULTIMO_MES)
# NDJ termina en enero del año siguiente
oni["Anio"] = oni["YR"] + (oni["Trimestre"] == "NDJ").astype(int)
oni["Fecha"] = pd.to_datetime(dict(year=oni["Anio"], month=oni["Mes"], day=1))

oni = oni.sort_values("Fecha").reset_index(drop=True)
oni[["Trimestre", "YR", "Fecha", "Anio", "Mes", "SST_Nino34", "ONI"]].iloc[8:16]

## 4. Validaciones

In [ ]:
print("Rango:", oni["Fecha"].min().date(), "a", oni["Fecha"].max().date())

# Debe haber exactamente un valor por mes, sin huecos (requisito para que los rezagos sean correctos)
assert oni["Fecha"].is_unique, "Hay meses duplicados"
meses_esperados = pd.date_range(oni["Fecha"].min(), oni["Fecha"].max(), freq="MS")
faltantes = meses_esperados.difference(oni["Fecha"])
assert len(faltantes) == 0, f"Meses faltantes: {list(faltantes)}"
assert oni["ONI"].notna().all()

print(f"{len(oni)} meses continuos, sin duplicados ni faltantes")

## 5. Rezagos y fase ENSO

- `ONI_lag1`, `ONI_lag3`, `ONI_lag6`, `ONI_lag12`: valor del ONI 1, 3, 6 y 12 meses antes. Se calculan sobre la serie
  mensual completa (antes de cruzar con SIPSA) para que los rezagos de 2013 usen datos de 2012.
- `Fase_ENSO`: clasificación mensual según el umbral de ±0.5 °C (`El Niño` ≥ 0.5, `La Niña` ≤ −0.5, `Neutral` en otro caso).
  Es una clasificación simplificada: la definición oficial de un *episodio* exige que el umbral se mantenga
  al menos 5 trimestres consecutivos.

In [ ]:
REZAGOS = [1, 3, 6, 12]

for k in REZAGOS:
    oni[f"ONI_lag{k}"] = oni["ONI"].shift(k)

oni["Fase_ENSO"] = pd.cut(
    oni["ONI"],
    bins=[-float("inf"), -0.5, 0.5 - 1e-9, float("inf")],
    labels=["La Niña", "Neutral", "El Niño"],
).astype(str)

columnas = ["Fecha", "Anio", "Mes", "Trimestre", "SST_Nino34", "ONI", "Fase_ENSO"] + [f"ONI_lag{k}" for k in REZAGOS]
oni = oni[columnas]
oni.tail(13)

In [ ]:
# Distribución de fases en el periodo de la serie SIPSA (2013 en adelante)
oni[oni["Anio"] >= 2013].groupby("Anio")["Fase_ENSO"].value_counts().unstack(fill_value=0)

## 6. Análisis de correlación cruzada (selección de rezagos)

Objetivo: documentar **qué rezagos del ONI se relacionan con los precios**, para justificar los rezagos que se exportan.

**Reglas para no contaminar la evaluación del modelo:**
- Se usa **solo el periodo de entrenamiento** (`FIN_ENTRENAMIENTO`). Si se eligieran los rezagos mirando también el
  periodo de prueba, se estaría filtrando información y el desempeño del modelo quedaría inflado.
  ⚠️ Ajustar `FIN_ENTRENAMIENTO` a la misma fecha de corte que usen en el modelo.

**Variable de precio:** variación anual del logaritmo del precio, Δ₁₂ log(P), calculada dentro de cada par
producto-mercado y luego promediada entre pares para cada mes. Así se elimina el nivel de precio de cada mercado,
la estacionalidad y la tendencia, y no influye que los mercados entren o salgan de la muestra.

**Pre-blanqueo (Box-Jenkins):** el ONI es una serie muy suave (está muy autocorrelacionada) y la variación anual del precio
también. Con dos series así, la correlación cruzada "cruda" sale alta en casi todos los rezagos aunque no haya relación
real, y las bandas de significancia habituales no son válidas. El pre-blanqueo corrige esto:
1. Se ajusta un modelo AR(p) al ONI (orden elegido por AIC, solo con datos hasta `FIN_ENTRENAMIENTO`).
2. Se aplica **el mismo filtro** al ONI y a la serie de precio.
3. Se calcula la correlación cruzada entre las dos series filtradas. Como el ONI filtrado es ruido blanco,
   la banda ±1.96/√n sí es válida (≈ 95 % de confianza).

Se evalúan los rezagos 0 a 18 meses: más amplio que los rezagos exportados, para comprobar que no se esté
dejando por fuera un rezago relevante.

In [ ]:
import numpy as np

SIPSA_FILE = os.path.join(OUT_DIR, "sipsa_precios_mayoristas_consolidado.csv")
FIN_ENTRENAMIENTO = "2022-12-01"   # ⚠️ usar la misma fecha de corte del modelo
REZAGOS_CCF = range(0, 19)

sipsa = pd.read_csv(SIPSA_FILE, parse_dates=["Fecha"], dtype={"Cod_departamento": str, "Cod_municipio": str})

# Series de precio a analizar: nombre -> filtro sobre los productos
SERIES = {
    "Arroz de primera": lambda d: d["Producto"] == "Arroz de primera",
    "Carne de res (todos los cortes)": lambda d: d["Producto"].str.startswith("Carne de res"),
    "Carne de cerdo (todos los cortes)": lambda d: d["Producto"].str.startswith("Carne de cerdo"),
    "Pollo entero sin vísceras": lambda d: d["Producto"] == "Pollo entero fresco sin vísceras",
}


def variacion_anual_log(d):
    """Δ12 log(precio) por par producto-mercado, promediada entre pares para cada mes."""
    pares = d.pivot_table(index="Fecha", columns=["Producto", "Mercado"], values="Precio_kg", aggfunc="mean")
    pares = pares.reindex(pd.date_range(pares.index.min(), pares.index.max(), freq="MS"))  # meses continuos
    log_p = np.log(pares)
    return (log_p - log_p.shift(12)).mean(axis=1, skipna=True)


precios = pd.DataFrame({nombre: variacion_anual_log(sipsa[filtro(sipsa)]) for nombre, filtro in SERIES.items()})
precios.index.name = "Fecha"
precios = precios.loc[:FIN_ENTRENAMIENTO].dropna(how="all")
print(f"Periodo de análisis (entrenamiento): {precios.index.min().date()} a {precios.index.max().date()} ({len(precios)} meses)")
precios.describe().round(3)

In [ ]:
def ajustar_ar(x, p_max=24):
    """Ajusta AR(p) por mínimos cuadrados y elige p por AIC. Devuelve (p, coeficientes phi_1..phi_p)."""
    x = np.asarray(x, dtype=float)
    y = x[p_max:]
    mejor = None
    for p in range(1, p_max + 1):
        X = np.column_stack([np.ones(len(y))] + [x[p_max - k:len(x) - k] for k in range(1, p + 1)])
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        resid = y - X @ beta
        aic = len(y) * np.log(resid.var()) + 2 * (p + 1)
        if mejor is None or aic < mejor[0]:
            mejor = (aic, p, beta[1:])
    return mejor[1], mejor[2]


def filtrar_ar(serie, phi):
    """Aplica el filtro (1 - phi_1 B - ... - phi_p B^p) a una serie mensual."""
    out = serie.copy()
    for k, coef in enumerate(phi, start=1):
        out = out - coef * serie.shift(k)
    return out


oni_mensual = oni.set_index("Fecha")["ONI"]
p, phi = ajustar_ar(oni_mensual.loc[:FIN_ENTRENAMIENTO])  # solo datos hasta el corte
print(f"Modelo de pre-blanqueo del ONI: AR({p})")

oni_blanco = filtrar_ar(oni_mensual, phi)

# Verificación: el ONI filtrado debe ser ruido blanco (si no, la banda ±1.96/√n no es válida)
residuos = oni_blanco.loc[:FIN_ENTRENAMIENTO].dropna()
acf_residuos = np.array([residuos.autocorr(k) for k in range(1, 25)])
fuera = int(np.sum(np.abs(acf_residuos) > 1.96 / np.sqrt(len(residuos))))
print(f"Autocorrelaciones de los residuos fuera de la banda: {fuera} de 24 (se esperan ~1 por azar)")
if fuera > 3:
    print("ADVERTENCIA: el ONI filtrado no es ruido blanco; aumentar p_max en ajustar_ar.")

filas = []
for nombre in precios.columns:
    y = precios[nombre].reindex(pd.date_range(precios.index.min(), precios.index.max(), freq="MS"))
    y_blanco = filtrar_ar(y, phi)
    for k in REZAGOS_CCF:
        # correlación entre el ONI de t-k y el precio de t
        cruda = pd.concat([oni_mensual.shift(k), y], axis=1, join="inner").dropna()
        blanca = pd.concat([oni_blanco.shift(k), y_blanco], axis=1, join="inner").dropna()
        n = len(blanca)
        filas.append({
            "Serie": nombre,
            "Rezago": k,
            "CCF_cruda": cruda.corr().iloc[0, 1],
            "CCF_preblanqueada": blanca.corr().iloc[0, 1],
            "Banda_95": 1.96 / np.sqrt(n),
            "n": n,
        })

ccf = pd.DataFrame(filas)
ccf["Significativo"] = ccf["CCF_preblanqueada"].abs() > ccf["Banda_95"]

# Tabla: correlación pre-blanqueada por rezago (* = significativa al 95 %)
tabla = ccf.assign(
    valor=ccf["CCF_preblanqueada"].round(2).astype(str) + np.where(ccf["Significativo"], "*", "")
).pivot(index="Rezago", columns="Serie", values="valor")[list(SERIES)]
tabla

In [ ]:
import matplotlib.pyplot as plt

AZUL, AZUL_CLARO = "#2a78d6", "#b7d3f6"
TINTA, TINTA_SECUNDARIA, BANDA = "#0b0b0b", "#52514e", "#f0efec"

fig, ejes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
for ax, nombre in zip(ejes.flat, SERIES):
    d = ccf[ccf["Serie"] == nombre]
    banda = d["Banda_95"].iloc[0]
    ax.axhspan(-banda, banda, color=BANDA, zorder=0)
    ax.axhline(0, color=TINTA_SECUNDARIA, linewidth=0.8, zorder=1)
    colores = np.where(d["Significativo"], AZUL, AZUL_CLARO)
    ax.bar(d["Rezago"], d["CCF_preblanqueada"], width=0.6, color=colores, zorder=2)
    for k in REZAGOS:  # rezagos exportados en el CSV
        ax.axvline(k, color=TINTA_SECUNDARIA, linewidth=0.6, linestyle=":", zorder=1)
    ax.set_title(nombre, loc="left", fontsize=11, color=TINTA)
    ax.tick_params(colors=TINTA_SECUNDARIA, labelsize=9)
    for lado in ["top", "right"]:
        ax.spines[lado].set_visible(False)
    for lado in ["left", "bottom"]:
        ax.spines[lado].set_color("#d0cfca")

for ax in ejes[1]:
    ax.set_xlabel("Rezago del ONI (meses)", color=TINTA_SECUNDARIA)
    ax.set_xticks(list(REZAGOS_CCF)[::3])
for ax in ejes[:, 0]:
    ax.set_ylabel("Correlación pre-blanqueada", color=TINTA_SECUNDARIA)

fig.suptitle(
    f"Correlación cruzada ONI(t−k) vs. variación anual del precio(t) — entrenamiento hasta {FIN_ENTRENAMIENTO[:7]}\n"
    "Azul oscuro = significativa al 95 % (fuera de la banda gris) · líneas punteadas = rezagos exportados (1, 3, 6, 12)",
    x=0.01, ha="left", fontsize=11, color=TINTA,
)
fig.tight_layout()
plt.show()

In [ ]:
# Resumen: rezagos significativos por serie (pre-blanqueados)
resumen_ccf = (
    ccf[ccf["Significativo"]]
    .groupby("Serie")
    .apply(lambda d: ", ".join(f"{k} ({c:+.2f})" for k, c in zip(d["Rezago"], d["CCF_preblanqueada"])), include_groups=False)
    .reindex(list(SERIES))
    .fillna("ninguno")
    .rename("Rezagos significativos (correlación)")
)

# Pruebas múltiples: con muchas pruebas al 95 %, algunas salen significativas solo por azar
n_pruebas = len(ccf)
n_signif = int(ccf["Significativo"].sum())
print(f"Significativos: {n_signif} de {n_pruebas} pruebas; por azar se esperan ~{0.05 * n_pruebas:.1f}")
print(f"Banda de significancia: ±{ccf['Banda_95'].iloc[0]:.3f} (n = {ccf['n'].iloc[0]} meses)")
resumen_ccf.to_frame()

### Cómo interpretar el resultado

- **Correlación cruda vs. pre-blanqueada:** la cruda (columna `CCF_cruda`) suele salir alta y suave en casi todos los
  rezagos. Eso no es evidencia de relación: es el efecto de dos series suaves con tendencias comunes. En el periodo de
  entrenamiento, por ejemplo, **La Niña 2020-2023 coincidió con el choque inflacionario pospandemia**, así que la
  correlación cruda mezcla ambos efectos. Para elegir rezagos se usa solo la pre-blanqueada.
- **Pruebas múltiples:** se hacen 4 series × 19 rezagos = 76 pruebas. Al 95 % se esperan ~4 significativas solo por azar.
  Un rezago aislado que apenas cruza la banda **no** es evidencia suficiente; uno que se repite en productos
  relacionados o que coincide con lo esperado agronómicamente sí lo es.
- **Si pocos rezagos son significativos:** con ~9 años de datos mensuales y precios promediados a nivel nacional,
  la potencia estadística es baja (solo hay 2-3 eventos ENSO en el periodo). En ese caso los rezagos 1, 3, 6 y 12 se
  justifican por **teoría** (ciclo del cultivo de arroz ≈ 4-6 meses; ciclo ganadero ≈ 6-12+ meses) y su utilidad se
  valida por **desempeño fuera de muestra** del modelo (en validación, no en prueba), no por esta correlación.
- **Siguiente paso posible:** repetir el análisis por departamento (p. ej. Tolima, Huila, Meta y Casanare para arroz),
  donde el efecto climático local puede ser más fuerte que en el promedio nacional.

## 7. Exportar CSV

In [ ]:
oni.to_csv(OUT_FILE, index=False, encoding="utf-8", date_format="%Y-%m-%d")
print(f"Archivo generado: {OUT_FILE} ({len(oni)} filas, {os.path.getsize(OUT_FILE) / 1024:.0f} KB)")

In [ ]:
# En Colab: descargar el CSV generado para subirlo al repositorio
if EN_COLAB:
    from google.colab import files
    files.download(OUT_FILE)

## 8. Uso: lectura desde GitHub y cruce con SIPSA

El cruce se hace por `Anio` y `Mes`. Para pronosticar el precio en t se usan las columnas `ONI_lag*`.

In [ ]:
BASE = "https://raw.githubusercontent.com/yevasa93/AnomaliasClimaticasArrozCarne/main/processed_data/"

oni = pd.read_csv(BASE + "oni_mensual.csv", parse_dates=["Fecha"])
sipsa = pd.read_csv(
    BASE + "sipsa_precios_mayoristas_consolidado.csv",
    parse_dates=["Fecha"],
    dtype={"Cod_departamento": str, "Cod_municipio": str},
)

datos = sipsa.merge(oni.drop(columns=["Fecha"]), on=["Anio", "Mes"], how="left")
print(datos.shape)
datos[["Fecha", "Producto", "Mercado", "Precio_kg", "ONI_lag1", "ONI_lag3", "ONI_lag6"]].head()